In [0]:
#API Ingestion and Bronze Layer
import requests
import json
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
# Creating catalog, schema and volume
spark.sql("CREATE CATALOG IF NOT EXISTS workspace" )
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.default")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.cricket_project")

base_path='/Volumes/workspace/default/cricket_project'

In [0]:
# Calling Cricket API
API_KEY='8acfbd55-4c06-4dcf-bc24-bdc34c01b4b6'
api_url=f"https://api.cricapi.com/v1/currentMatches?apikey=8acfbd55-4c06-4dcf-bc24-bdc34c01b4b6&offset=0"

response=requests.get(api_url)
response.raise_for_status

api_data=response.json()
print(api_data.keys())

print(json.dumps(api_data,indent=2)[:2000])

In [0]:
# Creating raw json data in volume
file_path=f"{base_path}/raw_data.json"
with open(file_path,'w') as file:
  json.dump(api_data,file)
print(" File saved at path: ",file_path)


In [0]:
# Creating the bronze layer Dataframe or Table
bronze_data=[{
    "source_api":api_url,
    "raw_json":json.dumps(api_data),
    "ingestion_time":None
}]

bronze_schema=StructType([
  StructField("source_api",StringType()),
  StructField("raw_json",StringType()),
  StructField("ingestion_time",TimestampType())
])

bronze_df=spark.createDataFrame(bronze_data,schema=bronze_schema)\
    .withColumn("ingestion_time",current_timestamp())

display(bronze_df)

In [0]:
# Saving The Bronze Table
bronze_df.write\
    .format('delta')\
    .mode('overwrite')\
    .saveAsTable("workspace.default.cricket_bronze_current")

print("Bronze Table created successfully.")